# ทำความสะอาดข้อมูล ขั้นที่ 1/5: ตรวจโครงสร้างและกำหนดขอบเขตข้อมูล

ต่อจาก `step1_questions.ipynb` ที่เราตั้งคำถามเรื่อง **ไฟฟ้าของไทยและอาเซียน** ไว้แล้ว ตอนนี้เริ่มทำความสะอาดข้อมูล ซึ่งแบ่งเป็น 5 ขั้น ตามเกณฑ์ข้อ **3.5.1 การเตรียมและจัดการข้อมูล (18 คะแนน)**

| ขั้น | เรื่อง | เกณฑ์ที่ตอบ |
|---|---|---|
| **1 (ไฟล์นี้)** | **ตรวจโครงสร้าง คีย์หลัก ข้อมูลซ้ำ ชนิดข้อมูล และกำหนดขอบเขต** | 3.5.1.1.1, 3.5.1.1.2, 3.5.1.1.5, 3.5.1.1.6 |
| 2 | จัดการค่าว่าง (Missing) | 3.5.1.1.4, 3.5.1.2.1 |
| 3 | ตรวจและจัดการค่าผิดปกติ (Outliers) | 3.5.1.2.2 |
| 4 | มาตรฐานรูปแบบ + ตรวจความสอดคล้องของตัวเลข | 3.5.1.1.3, 3.5.1.2.3, 3.5.1.2.4 |
| 5 | Data Dictionary + ส่งออกไฟล์สะอาด + สรุปขั้นตอน | 3.5.1.3 |

**ผลลัพธ์ของไฟล์นี้** (จะถูกบันทึกในโฟลเดอร์ `clean_output/`):
- `clean_step1.csv` ข้อมูลหลังตัดขอบเขต ใช้เป็นอินพุตของขั้นที่ 2
- `profile_raw.csv` รายงานสรุปข้อมูลดิบ (หลักฐาน "ก่อน" ทำความสะอาด)
- `cleaning_log.csv` บันทึกทุกการเปลี่ยนแปลง (จำนวนแถว/คอลัมน์ ก่อน–หลัง) ซึ่งจะเพิ่มต่อไปทุกขั้น

> **วิธีรัน:** แตกไฟล์ zip แล้วเปิดโน้ตบุ๊กนี้จากโฟลเดอร์ `energy_project` (ไฟล์ CSV อยู่ในโฟลเดอร์เดียวกันแล้ว)
> รันเรียงจากบนลงล่างด้วย **Shift + Enter** หรือใช้เมนู **Kernel → Restart Kernel and Run All Cells** ห้ามรันข้ามเซลล์

## 1.1 เตรียมเครื่องมือและโหลดข้อมูลดิบ
เราจะเก็บข้อมูลดิบไว้ในตัวแปร `raw` และ **ไม่แก้ไขมันเลย** งานทั้งหมดทำกับสำเนา `df` เพื่อให้ย้อนกลับมาเทียบได้เสมอ

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# หาไฟล์ข้อมูลอัตโนมัติ (รองรับชื่อที่ถูกเปลี่ยนตอนดาวน์โหลด เช่น World_Energy_Consumption (1).csv)
import glob
found = sorted(glob.glob("World_Energy_Consumption*.csv"))
if not found:
    raise FileNotFoundError(
        "ไม่พบไฟล์ World_Energy_Consumption.csv ในโฟลเดอร์ " + os.getcwd() +
        "\nให้วางไฟล์ CSV ไว้โฟลเดอร์เดียวกับโน้ตบุ๊กนี้ แล้วรันใหม่")
RAW_FILE = found[0]
print("ใช้ไฟล์ข้อมูล:", RAW_FILE)

OUT_DIR = "clean_output"
os.makedirs(OUT_DIR, exist_ok=True)

# บันทึกเวอร์ชันเครื่องมือไว้ (เกณฑ์ 3.5.1.3.2 ต้องระบุ dependency/เวอร์ชัน)
print("Python    :", sys.version.split()[0])
print("pandas    :", pd.__version__)
print("numpy     :", np.__version__)
print("matplotlib:", matplotlib.__version__)

raw = pd.read_csv(RAW_FILE)
df = raw.copy()
print("\nข้อมูลดิบ: ", raw.shape[0], "แถว x", raw.shape[1], "คอลัมน์")

### ตัวช่วยบันทึก log
ทุกครั้งที่ตัดแถวหรือคอลัมน์ เราจะเรียก `log_step()` เพื่อจดว่า **ทำอะไร ก่อนกี่แถว หลังกี่แถว เพราะอะไร**
กรรมการจะเห็นร่องรอยที่ตรวจสอบย้อนกลับได้ (traceability) ซึ่งเกณฑ์ให้คะแนนโดยตรง

In [ ]:
LOG_FILE = os.path.join(OUT_DIR, "cleaning_log.csv")
log_rows = []   # ไฟล์นี้เป็นขั้นแรก จึงเริ่ม log ใหม่

def log_step(action, before, after, reason):
    """before/after = DataFrame ก่อนและหลังทำ"""
    log_rows.append({
        "step": "1-structure",
        "action": action,
        "rows_before": before.shape[0], "rows_after": after.shape[0],
        "cols_before": before.shape[1], "cols_after": after.shape[1],
        "reason": reason,
    })
    print(f"✔ {action}: แถว {before.shape[0]:,} → {after.shape[0]:,} | คอลัมน์ {before.shape[1]} → {after.shape[1]}")

## 1.2 Profiling ข้อมูลดิบ (หลักฐาน "ก่อน")
เกณฑ์ข้อ 3.5.1.1 ต้องการ summary: **shape, dtypes, %missing/column, duplicate rate**
เราสร้างตารางสรุปทุกคอลัมน์ไว้ แล้วบันทึกเป็น `profile_raw.csv`

In [ ]:
def profile(data):
    """สรุปทุกคอลัมน์: ชนิดข้อมูล, จำนวนค่าว่าง, %ว่าง, จำนวนค่าไม่ซ้ำ, ตัวอย่างค่า, min, max"""
    out = pd.DataFrame({
        "dtype": data.dtypes.astype(str),
        "non_null": data.notna().sum(),
        "missing": data.isna().sum(),
        "missing_pct": (data.isna().mean() * 100).round(2),
        "n_unique": data.nunique(),
        "example": [data[c].dropna().iloc[0] if data[c].notna().any() else None for c in data.columns],
    })
    num = data.select_dtypes("number")
    out["min"] = num.min()
    out["max"] = num.max()
    return out

prof_raw = profile(raw)
prof_raw.to_csv(os.path.join(OUT_DIR, "profile_raw.csv"), encoding="utf-8-sig")

print("ชนิดข้อมูลที่พบ:")
print(raw.dtypes.astype(str).value_counts())
print("\n% ค่าว่างเฉลี่ยทั้งไฟล์:", round(raw.isna().mean().mean() * 100, 1), "%")
prof_raw.head(15)

In [ ]:
# ภาพรวมค่าว่างทั้งไฟล์ (missing map): สีเข้ม = ว่าง
plt.figure(figsize=(12, 4))
plt.imshow(raw.isna().values[::20].T, aspect="auto", cmap="Greys", interpolation="nearest")
plt.yticks([]); plt.xlabel("rows (every 20th row)"); plt.ylabel(f"{raw.shape[1]} columns")
plt.title("Missing map of RAW data (dark = missing)")
plt.show()

> **สังเกต:** ไฟล์ดิบว่างเกินครึ่ง สาเหตุหลักคือมีปี 1900–1964 ที่แทบไม่มีข้อมูล และมีคอลัมน์ที่ใช้ไม่ได้จริงจำนวนมาก นี่คือเหตุผลที่ต้อง **กำหนดขอบเขต** ก่อนจัดการค่าว่าง (ถ้าไปเติมค่าว่างทั้งไฟล์เลยจะผิดวิธี)

## 1.3 ตรวจคีย์หลัก (Primary Key) และข้อมูลซ้ำ
ตารางนี้ **1 แถว = 1 ประเทศ ใน 1 ปี** ดังนั้นคีย์หลักคือ `country + year`
เช็ก 3 อย่าง:
1. คีย์ห้ามว่าง
2. คีย์ห้ามซ้ำ
3. ชื่อประเทศไม่มีช่องว่างเกิน/ตัวพิมพ์ต่างกัน (ถ้ามี จะทำให้ "Thailand" กับ "Thailand " ถูกนับเป็นคนละประเทศ ทั้งที่จริงคือแถวซ้ำแอบแฝง)

In [ ]:
KEY = ["country", "year"]

print("1) คีย์ว่าง          :", df[KEY].isna().any(axis=1).sum(), "แถว")
print("2) คีย์ซ้ำ           :", df.duplicated(KEY).sum(), "แถว")
print("   แถวซ้ำทั้งแถว      :", df.duplicated().sum(), "แถว")

# 3) ทำชื่อให้เป็นมาตรฐาน แล้วดูว่ายังซ้ำไหม
name_std = df["country"].str.strip().str.replace(r"\s+", " ", regex=True)
print("3) ชื่อที่มีช่องว่างเกิน:", (name_std != df["country"]).sum(), "แถว")
print("   ชื่อซ้ำเมื่อไม่สนตัวพิมพ์:", df.assign(k=name_std.str.lower()).duplicated(["k", "year"]).sum(), "แถว")

In [ ]:
# จัดการ (ถึงแม้ผลตรวจจะเป็น 0 ก็ควรใส่โค้ดไว้ เพื่อให้ pipeline ใช้ได้กับข้อมูลชุดใหม่)
before = df
df = df.assign(country=name_std)
df = df.drop_duplicates(KEY, keep="first")
log_step("strip ชื่อประเทศ + ลบแถวซ้ำบนคีย์ country+year", before, df,
         "คีย์หลักต้องไม่ซ้ำ; ผลตรวจพบซ้ำ 0 แถว")

dup_rate_before = raw.duplicated(KEY).mean() * 100
dup_rate_after = df.duplicated(KEY).mean() * 100
print(f"อัตราซ้ำ ก่อน {dup_rate_before:.2f}% → หลัง {dup_rate_after:.2f}%")

## 1.4 ตรวจชนิดข้อมูล (dtypes)
ชนิดที่ควรเป็น:
- `year` → **จำนวนเต็ม** (ไม่ใช่ทศนิยม หรือข้อความ)
- `country`, `iso_code` → **ข้อความ/หมวดหมู่**
- คอลัมน์อื่นทั้งหมด → **ตัวเลข (float)**

ถ้ามีคอลัมน์ตัวเลขที่ pandas อ่านเป็นข้อความ (เช่นมีคำว่า "n/a" หรือ "1,234" ปนอยู่) ต้องหาให้เจอ

In [ ]:
TEXT_COLS = ["country", "iso_code"]
num_cols = [c for c in df.columns if c not in TEXT_COLS + ["year"]]

wrong_type = [c for c in num_cols if not pd.api.types.is_numeric_dtype(df[c])]
print("คอลัมน์ที่ควรเป็นตัวเลขแต่ไม่ใช่:", wrong_type if wrong_type else "ไม่มี ✅")
print("year เป็นจำนวนเต็ม:", pd.api.types.is_integer_dtype(df["year"]))

# ถ้ามีคอลัมน์ผิดชนิด แปลงด้วย to_numeric แล้วนับว่ามีค่าที่แปลงไม่ได้กี่ค่า
for c in wrong_type:
    converted = pd.to_numeric(df[c], errors="coerce")
    bad = converted.isna() & df[c].notna()
    print(f"  {c}: แปลงไม่ได้ {bad.sum()} ค่า เช่น {df.loc[bad, c].unique()[:5]}")
    df[c] = converted

## 1.5 แยกประเทศจริงออกจากกลุ่มภูมิภาค (Referential Integrity)
ไฟล์นี้ไม่มีตารางที่สองให้ทำ Foreign Key โดยตรง เราจึงใช้ **`iso_code` เป็นเหมือน FK ไปยัง "ทะเบียนประเทศ"** แล้วเช็กว่า
- 1 ประเทศ ต้องมี iso_code เดียว และ 1 iso_code ต้องเป็นของประเทศเดียว (ความสัมพันธ์ 1:1)
- รหัสต้องยาว 3 ตัวอักษรตามมาตรฐาน ISO 3166-1 alpha-3

ส่วนชื่อที่ไม่มี iso_code แบ่งเป็น 3 พวก:
1. **กลุ่ม/ภูมิภาค** เช่น Asia, OECD (EI) → ตัดออก ไม่งั้นรวมยอดแล้ว **นับซ้ำ**
2. **ประเทศที่ไม่มีแล้ว** เช่น USSR, Czechoslovakia → ตัดออก เพราะมีประเทศใหม่ที่แยกออกมาแทนแล้ว (นับซ้ำเช่นกัน)
3. **ประเทศจริงแต่ไม่มีรหัส** คือ Kosovo → เก็บไว้ และใส่รหัส `XKX` (รหัสที่หน่วยงานสากลใช้กับ Kosovo)

In [ ]:
# ความสัมพันธ์ 1:1 ระหว่าง country กับ iso_code
has_code = df["iso_code"].notna()
codes_per_country = df[has_code].groupby("country")["iso_code"].nunique()
countries_per_code = df[has_code].groupby("iso_code")["country"].nunique()
bad_len = df.loc[has_code & (df["iso_code"].str.len() != 3), "iso_code"].unique()

print("ประเทศที่มีมากกว่า 1 รหัส :", (codes_per_country > 1).sum())
print("รหัสที่ใช้กับมากกว่า 1 ประเทศ:", (countries_per_code > 1).sum())
print("รหัสที่ไม่ใช่ 3 ตัวอักษร   :", list(bad_len))

In [ ]:
# จัดประเภทชื่อที่ไม่มี iso_code
REAL_NO_CODE = {"Kosovo": "XKX"}                       # ประเทศจริง เติมรหัส
HISTORICAL = ["USSR", "Czechoslovakia", "Serbia and Montenegro",
              "East Germany (EIA)", "Yugoslavia"]        # ประเทศที่ไม่มีแล้ว

no_code_names = sorted(df.loc[~has_code, "country"].unique())
aggregates = [n for n in no_code_names if n not in REAL_NO_CODE and n not in HISTORICAL]
print(f"ไม่มี iso_code ทั้งหมด {len(no_code_names)} ชื่อ")
print(f"  - ประเทศจริงที่เติมรหัส : {list(REAL_NO_CODE)}")
print(f"  - ประเทศที่ไม่มีแล้ว    : {[n for n in HISTORICAL if n in no_code_names]}")
print(f"  - กลุ่ม/ภูมิภาค         : {len(aggregates)} ชื่อ เช่น {aggregates[:6]}")

In [ ]:
before = df
df = df.copy()
for name, code in REAL_NO_CODE.items():
    df.loc[df["country"] == name, "iso_code"] = code
df = df[df["iso_code"].notna()].copy()
log_step("ตัดกลุ่มภูมิภาค/ประเทศที่ไม่มีแล้ว, เติม iso_code ให้ Kosovo", before, df,
         "ป้องกันการนับซ้ำเมื่อรวมยอด; เก็บเฉพาะประเทศจริง")

# FK match rate: ทุกแถวที่เหลือต้องมีรหัสที่ถูกต้อง (เกณฑ์ต้องการ ≥ 99%)
fk_ok = df["iso_code"].str.fullmatch(r"[A-Z]{3}")
print(f"FK match rate (iso_code ถูกรูปแบบ): {fk_ok.mean()*100:.2f}%")
print("จำนวนประเทศ:", df["country"].nunique())

## 1.6 กำหนดขอบเขตช่วงปี (ด้วยหลักฐาน ไม่ใช่เดา)
คำถามของเราเกี่ยวกับ **ไฟฟ้า** จึงดูว่าแต่ละปี มีกี่ประเทศที่มีข้อมูล `electricity_generation`
แล้วเลือกช่วงปีที่ข้อมูลครอบคลุมเกือบทุกประเทศ

In [ ]:
n_countries = df["country"].nunique()
cov = (df.groupby("year")[["electricity_generation", "primary_energy_consumption", "population"]]
         .count() / n_countries * 100)

ax = cov.loc[1965:].plot(figsize=(10, 3.5), marker=".")
ax.axvline(2000, color="red", ls="--"); ax.axhline(90, color="grey", ls=":")
ax.set_ylabel("% of countries with data"); ax.set_ylim(0, 105)
ax.set_title("Data coverage by year (red line = 2000)")
plt.show()

cov.loc[[1985, 1990, 1995, 1999, 2000, 2010, 2020, 2021, 2022]].round(1)

**อ่านกราฟ:**
- ก่อนปี 2000 มีข้อมูลไฟฟ้าแค่ราว 1 ใน 3 ของประเทศ ถ้าเอามาใช้ ค่าเฉลี่ยโลกจะเพี้ยน เพราะประเทศที่มีข้อมูลเป็นประเทศรวยเกือบทั้งหมด
- ตั้งแต่ **ปี 2000** ครอบคลุมเกือบทุกประเทศ
- ปี **2022 ข้อมูลยังไม่ครบ** (มีไม่ถึงครึ่ง) แต่ยังเก็บไว้ เพราะไทยและอาเซียนบางประเทศมีข้อมูล เราจะติดป้าย "ปีข้อมูลไม่ครบ" ในขั้นที่ 4 (เรื่องความเป็นปัจจุบัน/timeliness)

**ตัดสินใจ:** ใช้ช่วงปี **2000–2022** (เปลี่ยนได้ที่ตัวแปร `YEAR_START`, `YEAR_END`)

In [ ]:
YEAR_START, YEAR_END = 2000, 2022

before = df
df = df[df["year"].between(YEAR_START, YEAR_END)].copy()
log_step(f"เลือกช่วงปี {YEAR_START}-{YEAR_END}", before, df,
         "ก่อนปี 2000 มีข้อมูลไฟฟ้าไม่ถึง 40% ของประเทศ")

## 1.7 เลือกคอลัมน์ที่ตอบโจทย์
จาก 129 คอลัมน์ เราเลือกเฉพาะที่คำถามและแดชบอร์ดต้องใช้ แบ่งเป็นกลุ่ม:
- **คีย์**: country, iso_code, year
- **บริบท**: population, gdp
- **ไฟฟ้าภาพรวม**: ผลิต, ต้องการ, นำเข้าสุทธิ, ต่อคน, ความเข้มคาร์บอน
- **ไฟฟ้าแยกเชื้อเพลิง**: ปริมาณ (TWh) และสัดส่วน (%) ของแต่ละเชื้อเพลิง
- **พลังงานรวม**: primary_energy_consumption, energy_per_capita

คอลัมน์ `*_change_pct` และ `*_change_twh` ไม่เลือก เพราะคำนวณใหม่เองได้จากปริมาณ (และคำนวณเองจะตรวจสอบได้มากกว่า)

In [ ]:
FUELS = ["coal", "gas", "oil", "nuclear", "hydro", "solar", "wind", "biofuel", "other_renewable_exc_biofuel"]
GROUPS = ["fossil", "renewables", "low_carbon"]

KEY_COLS = ["country", "iso_code", "year"]
CONTEXT_COLS = ["population", "gdp"]
ELEC_COLS = ["electricity_generation", "electricity_demand", "net_elec_imports",
             "per_capita_electricity", "carbon_intensity_elec", "greenhouse_gas_emissions"]
ENERGY_COLS = ["primary_energy_consumption", "energy_per_capita"]
FUEL_TWH = [f"{f}_electricity" for f in FUELS + GROUPS]
FUEL_SHARE = [f"{f}_share_elec" for f in ["coal", "gas", "oil", "nuclear", "hydro", "solar", "wind", "biofuel"] + GROUPS]
FUEL_SHARE += ["other_renewables_share_elec_exc_biofuel"]   # ชื่อคอลัมน์จริงในไฟล์ต่างจากแบบแผนเล็กน้อย

SELECTED = KEY_COLS + CONTEXT_COLS + ELEC_COLS + ENERGY_COLS + FUEL_TWH + FUEL_SHARE

missing_cols = [c for c in SELECTED if c not in df.columns]
print("คอลัมน์ที่เลือกแต่ไม่มีในไฟล์:", missing_cols if missing_cols else "ไม่มี ✅")

before = df
df = df[SELECTED].copy()
log_step("เลือกคอลัมน์ที่ตอบโจทย์ไฟฟ้า/พลังงาน", before, df,
         "ตัดคอลัมน์ที่ไม่เกี่ยวกับคำถาม และ *_change_* ที่คำนวณใหม่ได้")

## 1.8 ปรับชนิดข้อมูลให้เหมาะกับการใช้งาน
- `country`, `iso_code` → `category` (ประหยัดหน่วยความจำ ทำให้แดชบอร์ดกรองเร็วขึ้น เกี่ยวกับเกณฑ์ Performance)
- `year` → `int` (เก็บเป็นปี ค.ศ.; ข้อมูลรายปีไม่จำเป็นต้องเป็น datetime)
- ตัวเลขอื่น → `float`

In [ ]:
mem_before = df.memory_usage(deep=True).sum() / 1024
df["country"] = df["country"].astype("category")
df["iso_code"] = df["iso_code"].astype("category")
df["year"] = df["year"].astype(int)
mem_after = df.memory_usage(deep=True).sum() / 1024

print(f"หน่วยความจำ: {mem_before:,.0f} KB → {mem_after:,.0f} KB")
df.dtypes.astype(str).value_counts()

## 1.9 สรุปผลขั้นที่ 1 และบันทึกไฟล์

In [ ]:
summary = pd.DataFrame({
    "ก่อน (ดิบ)": [raw.shape[0], raw.shape[1], raw["country"].nunique(),
                  f"{raw['year'].min()}-{raw['year'].max()}",
                  round(raw.duplicated(KEY).mean()*100, 2), round(raw.isna().mean().mean()*100, 1)],
    "หลังขั้นที่ 1": [df.shape[0], df.shape[1], df["country"].nunique(),
                    f"{df['year'].min()}-{df['year'].max()}",
                    round(df.duplicated(KEY).mean()*100, 2), round(df.isna().mean().mean()*100, 1)],
}, index=["จำนวนแถว", "จำนวนคอลัมน์", "จำนวนประเทศ/ชื่อ", "ช่วงปี", "อัตราคีย์ซ้ำ (%)", "ค่าว่างเฉลี่ย (%)"])
summary

In [ ]:
# % ค่าว่างของแต่ละคอลัมน์ที่เหลือ (จะจัดการในขั้นที่ 2)
miss = (df.isna().mean() * 100).sort_values(ascending=False)
miss[miss > 0].plot.barh(figsize=(8, 8), title="Missing % per column after step 1")
plt.axvline(5, color="red", ls="--"); plt.xlabel("% missing (red = 5% threshold)")
plt.gca().invert_yaxis(); plt.show()

In [ ]:
df.to_csv(os.path.join(OUT_DIR, "clean_step1.csv"), index=False, encoding="utf-8-sig")
pd.DataFrame(log_rows).to_csv(LOG_FILE, index=False, encoding="utf-8-sig")

print("บันทึกแล้ว:")
for f in sorted(os.listdir(OUT_DIR)):
    print("  -", f, f"({os.path.getsize(os.path.join(OUT_DIR, f))/1024:,.0f} KB)")
pd.DataFrame(log_rows)

## ✅ เช็กลิสต์ก่อนไปขั้นที่ 2
- [ ] รันครบทุกเซลล์โดยไม่มี error
- [ ] มีไฟล์ 3 ไฟล์ในโฟลเดอร์ `clean_output/`
- [ ] อธิบายได้ว่า **ทำไมตัดกลุ่มภูมิภาคออก** (ป้องกันนับซ้ำ) และ **ทำไมเริ่มปี 2000** (ดูจากกราฟ coverage) เพราะกรรมการมักถามตอน Q&A
- [ ] ดูกราฟสุดท้ายแล้วจดไว้ว่าคอลัมน์ไหนว่างเกิน 5% เพราะเป็นงานของขั้นที่ 2

**ขั้นถัดไป (ไฟล์ที่ 2):** จัดการค่าว่าง ทั้งแยก "ว่างเพราะไม่มีจริง" กับ "ว่างเพราะไม่ได้เก็บ" เติมค่าด้วยวิธีที่เหมาะสม และแสดงผลกระทบต่อสถิติก่อน–หลัง